# Task C Quantitative Metrics Contracts (Standalone v2)

**Mục tiêu:** khóa và kiểm thử ba hợp đồng định lượng: repeatability, similarity và co-contraction supportability.

**Ranh giới an toàn**
- Không dùng overlapping windows như repetitions độc lập.
- Không pooled Mendeley + GRABMyo.
- Không tạo hard fatigue diagnosis, pathology claim hoặc treatment recommendation.
- Co-contraction phải fail-closed nếu chưa xác minh anatomical mapping/synchronization/normalization.

**Input**
- `day36-final-manifest.json` trong real mode.
- Một NPZ feature artifact của Mendeley hoặc GRABMyo.
- Optional verified muscle-pair mapping + synchronized envelopes.

**Output**
- metric registry;
- repetition feature table;
- repeatability/similarity results;
- explicit co-contraction eligibility;
- Day36 integration evidence;
- final manifest + handoff ZIP.

## Cell 1 — Environment

**Input:** Colab/Python runtime.  
**Output:** deterministic environment and helper packages.

In [1]:
from __future__ import annotations

import hashlib
import itertools
import json
import math
import os
import platform
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler

SEED = 3701
rng = np.random.default_rng(SEED)

print({
    'python': platform.python_version(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'seed': SEED,
})

{'python': '3.12.13', 'numpy': '2.0.2', 'pandas': '2.2.2', 'seed': 3701}


## Cell 2 — Configuration and governance

**Input:** dataset profile and Drive paths.  
**Output:** frozen run configuration.

In [2]:
RUN_MODE = os.getenv('DAY37_RUN_MODE', 'MENDELEY').upper()
assert RUN_MODE in {'SYNTHETIC', 'MENDELEY', 'GRABMYO'}

SEALED_TEST_OPENED = False
POOLED_DATASETS_ALLOWED = False
HARD_FATIGUE_DIAGNOSIS_ALLOWED = False
TREATMENT_RECOMMENDATION_ALLOWED = False

assert SEALED_TEST_OPENED is False
assert POOLED_DATASETS_ALLOWED is False
assert HARD_FATIGUE_DIAGNOSIS_ALLOWED is False
assert TREATMENT_RECOMMENDATION_ALLOWED is False

DRIVE_ROOT = Path('/content/drive/MyDrive/MyoLab-AI-data')
PROFILE = {
    'SYNTHETIC': {
        'dataset_id': 'synthetic-taskc-contract',
        'npz': None,
        'day36_manifest': None,
        'output_dir': Path('/content/day37-taskc-synthetic-v2'),
    },
    'MENDELEY': {
        'dataset_id': 'mendeley-4channel-hand-gesture-v2',
        'npz': DRIVE_ROOT / 'mendeley-4channel-hand-gesture-v2/outputs/day31-mendeley-primary-fall14.npz',
        'day36_manifest': DRIVE_ROOT / 'mendeley-4channel-hand-gesture-v2/outputs/day36-context/day36-final-manifest.json',
        'output_dir': DRIVE_ROOT / 'mendeley-4channel-hand-gesture-v2/outputs/day37-taskc/day37-taskc-v2',
    },
    'GRABMYO': {
        'dataset_id': 'grabmyo-physionet-v1.1.0',
        'npz': DRIVE_ROOT / 'grabmyo-physionet-v1.1.0/outputs/grabmyo-primary4-forearm16-fall14.npz',
        'day36_manifest': DRIVE_ROOT / 'grabmyo-physionet-v1.1.0/outputs/day36-context/day36-final-manifest.json',
        'output_dir': DRIVE_ROOT / 'grabmyo-physionet-v1.1.0/outputs/day37-taskc/day37-taskc-v2',
    },
}[RUN_MODE]

OUTPUT_DIR = PROFILE['output_dir']
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    'registry': OUTPUT_DIR / 'taskc-metric-registry.yaml',
    'repetition_table': OUTPUT_DIR / 'taskc-repetition-features.parquet',
    'repetition_manifest': OUTPUT_DIR / 'taskc-repetition-feature-manifest.json',
    'repeatability': OUTPUT_DIR / 'repeatability-results.csv',
    'repeatability_summary': OUTPUT_DIR / 'repeatability-subject-summary.csv',
    'icc': OUTPUT_DIR / 'repeatability-icc-results.csv',
    'bland_altman': OUTPUT_DIR / 'between-session-bland-altman.csv',
    'similarity': OUTPUT_DIR / 'similarity-results.csv',
    'similarity_summary': OUTPUT_DIR / 'similarity-subject-summary.csv',
    'cocontraction_eligibility': OUTPUT_DIR / 'cocontraction-eligibility.json',
    'cocontraction_results': OUTPUT_DIR / 'cocontraction-results.csv',
    'sensitivity': OUTPUT_DIR / 'taskc-sensitivity-report.json',
    'integration': OUTPUT_DIR / 'day37-day36-integration-report.json',
    'final': OUTPUT_DIR / 'day37-final-manifest.json',
    'handoff': OUTPUT_DIR / 'day37-taskc-handoff.zip',
}

print(json.dumps({
    'run_mode': RUN_MODE,
    'dataset_id': PROFILE['dataset_id'],
    'npz': str(PROFILE['npz']) if PROFILE['npz'] else None,
    'output_dir': str(OUTPUT_DIR),
}, indent=2))

{
  "run_mode": "SYNTHETIC",
  "dataset_id": "synthetic-taskc-contract",
  "npz": null,
  "output_dir": "/content/day37-taskc-synthetic-v2"
}


## Cell 3 — Metric helpers and supportability states

**Input:** numeric arrays.  
**Output:** safe metrics with explicit reason codes.

In [3]:
EPS = 1e-12
CANONICAL_FEATURES = [
    'MAV', 'RMS', 'WL', 'ZC', 'SSC', 'WAMP', 'VAR', 'IEMG',
    'MNF', 'MDF', 'PKF', 'SM1', 'SM2', 'SM3',
]

def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

def json_safe(value: Any) -> Any:
    if isinstance(value, Path): return str(value)
    if isinstance(value, np.generic): return value.item()
    if isinstance(value, np.ndarray): return value.tolist()
    if isinstance(value, dict): return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [json_safe(v) for v in value]
    return value

def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), ensure_ascii=False, indent=2), encoding='utf-8')

def absolute_difference(x1: float, x2: float) -> float:
    return float(abs(x1 - x2))

def symmetric_relative_difference(x1: float, x2: float, eps: float = EPS):
    denom = abs(x1) + abs(x2)
    if denom <= eps:
        return None, 'DENOMINATOR_NEAR_ZERO'
    return float(2.0 * abs(x1 - x2) / (denom + eps)), 'SUPPORTED'

def within_subject_cv(values: Iterable[float], ratio_scale_positive: bool, eps: float = EPS):
    x = np.asarray(list(values), dtype=float)
    if x.size < 3:
        return None, 'INSUFFICIENT_REPETITIONS'
    if not ratio_scale_positive:
        return None, 'INELIGIBLE_NON_RATIO_SCALE'
    mean = float(np.mean(x))
    if mean <= eps or np.any(x < 0):
        return None, 'MEAN_NONPOSITIVE_OR_VALUES_NEGATIVE'
    return float(100.0 * np.std(x, ddof=1) / mean), 'SUPPORTED'

def robust_mad_ratio(values: Iterable[float], eps: float = EPS):
    x = np.asarray(list(values), dtype=float)
    if x.size < 3:
        return None, 'INSUFFICIENT_REPETITIONS'
    med = float(np.median(x))
    if abs(med) <= eps:
        return None, 'MEDIAN_NEAR_ZERO'
    mad = float(np.median(np.abs(x - med)))
    return float(1.4826 * mad / (abs(med) + eps)), 'SUPPORTED'

def bland_altman(x: Iterable[float], y: Iterable[float]) -> dict[str, Any]:
    a = np.asarray(list(x), dtype=float)
    b = np.asarray(list(y), dtype=float)
    if a.shape != b.shape or a.size < 3:
        return {'supportability': 'INSUFFICIENT_PAIRED_MEASUREMENTS', 'n': int(min(a.size, b.size))}
    d = b - a
    bias = float(np.mean(d))
    sd = float(np.std(d, ddof=1))
    return {
        'supportability': 'SUPPORTED',
        'n': int(d.size),
        'bias': bias,
        'loa_low': float(bias - 1.96 * sd),
        'loa_high': float(bias + 1.96 * sd),
    }

def cosine_similarity_safe(x: np.ndarray, y: np.ndarray, eps: float = EPS):
    nx, ny = float(np.linalg.norm(x)), float(np.linalg.norm(y))
    if nx <= eps or ny <= eps:
        return None, 'ZERO_NORM_VECTOR'
    return float(np.dot(x, y) / (nx * ny)), 'SUPPORTED'

def pearson_similarity_safe(x: np.ndarray, y: np.ndarray, eps: float = EPS):
    if np.std(x) <= eps or np.std(y) <= eps:
        return None, 'CONSTANT_VECTOR'
    return float(np.corrcoef(x, y)[0, 1]), 'SUPPORTED'

def normalized_euclidean(x: np.ndarray, y: np.ndarray) -> float:
    return float(np.linalg.norm(x - y) / math.sqrt(x.size))


def icc_a1_point(matrix: np.ndarray, eps: float = EPS):
    """Two-way random, absolute-agreement, single-measure ICC(A,1).

    Rows are subjects/targets; columns are repeated measurements/raters.
    Returns (value, supportability).
    """
    x = np.asarray(matrix, dtype=float)
    if x.ndim != 2 or x.shape[0] < 3 or x.shape[1] < 2 or not np.isfinite(x).all():
        return None, 'INSUFFICIENT_OR_INVALID_BALANCED_DESIGN'
    n, k = x.shape
    grand = float(np.mean(x))
    row_means = np.mean(x, axis=1)
    col_means = np.mean(x, axis=0)
    ss_rows = k * np.sum((row_means - grand) ** 2)
    ss_cols = n * np.sum((col_means - grand) ** 2)
    ss_total = np.sum((x - grand) ** 2)
    ss_error = ss_total - ss_rows - ss_cols
    ms_rows = ss_rows / (n - 1)
    ms_cols = ss_cols / (k - 1)
    ms_error = ss_error / ((n - 1) * (k - 1))
    denom = ms_rows + (k - 1) * ms_error + (k * (ms_cols - ms_error) / n)
    if abs(denom) <= eps:
        return None, 'ICC_DENOMINATOR_NEAR_ZERO'
    return float((ms_rows - ms_error) / denom), 'SUPPORTED'

def icc_subject_bootstrap_ci(matrix: np.ndarray, n_boot: int = 100, alpha: float = 0.05):
    x = np.asarray(matrix, dtype=float)
    point, state = icc_a1_point(x)
    if state != 'SUPPORTED':
        return point, None, None, state
    values = []
    local_rng = np.random.default_rng(SEED + 17)
    for _ in range(n_boot):
        idx = local_rng.integers(0, x.shape[0], size=x.shape[0])
        v, s = icc_a1_point(x[idx])
        if s == 'SUPPORTED' and np.isfinite(v):
            values.append(v)
    if len(values) < max(50, n_boot // 5):
        return point, None, None, 'BOOTSTRAP_CI_UNSTABLE'
    low, high = np.quantile(values, [alpha/2, 1-alpha/2])
    return point, float(low), float(high), 'SUPPORTED_BOOTSTRAP_SUBJECT_CI'

def cocontraction_metrics(a, b, dt, threshold_a, threshold_b, eps=EPS):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if a.shape != b.shape or a.ndim != 1 or a.size < 2:
        raise ValueError('Envelopes must be synchronized equal-length 1-D arrays.')
    if np.any(a < 0) or np.any(b < 0) or not np.isfinite(a).all() or not np.isfinite(b).all():
        raise ValueError('Envelopes must be finite and non-negative.')
    inst = 2.0 * np.minimum(a, b) / (a + b + eps)
    overlap = np.trapz(np.minimum(a, b), dx=dt)
    union = np.trapz(np.maximum(a, b), dx=dt)
    return {
        'mean_cci': float(np.mean(inst)),
        'median_cci': float(np.median(inst)),
        'q90_cci': float(np.quantile(inst, 0.90)),
        'overlap_area_ratio': float(overlap / (union + eps)),
        'simultaneous_activation_duration_fraction': float(np.mean((a > threshold_a) & (b > threshold_b))),
        'coactivation_load': float(overlap),
        'active_duration_seconds': float(a.size * dt),
    }

print('[PASS] Metric helpers loaded.')

[PASS] Metric helpers loaded.


## Cell 4 — Day 36 gate and dataset loading

**Input:** Day 36 manifest + NPZ or synthetic fixture.  
**Output:** window-level metadata and feature matrix.

In [4]:
def make_synthetic_window_data():
    subjects = [f'S{i:02d}' for i in range(1, 13)]
    classes = ['rest', 'hand_close', 'wrist_flexion', 'wrist_extension']
    sessions = ['D1', 'D2']
    feature_names = [f'CH{c+1}__{f}' for c in range(3) for f in CANONICAL_FEATURES]
    rows, vectors = [], []
    for s_idx, subject in enumerate(subjects):
        split = 'train' if s_idx < 8 else 'validation'
        subject_shift = rng.normal(0, 0.15, len(feature_names))
        for session_idx, session in enumerate(sessions):
            session_shift = rng.normal(0, 0.05 + 0.02 * session_idx, len(feature_names))
            for class_idx, label in enumerate(classes):
                class_center = np.zeros(len(feature_names))
                class_center[class_idx::4] = 0.8
                for rep in range(1, 6):
                    rep_id = f'{subject}-{session}-{label}-R{rep:02d}'
                    rep_center = class_center + subject_shift + session_shift + rng.normal(0, 0.08, len(feature_names))
                    for w in range(12):
                        vectors.append(rep_center + rng.normal(0, 0.06, len(feature_names)))
                        rows.append({
                            'dataset_id': 'synthetic-taskc-contract',
                            'subject_id': subject,
                            'session_id': session,
                            'protocol_id': 'synthetic-gesture-v1',
                            'protocol_version': '1.0.0',
                            'task_label': label,
                            'repetition_id': rep_id,
                            'window_id': f'{rep_id}-W{w:03d}',
                            'split_name': split,
                            'quality_status': 'pass',
                        })
    return pd.DataFrame(rows), np.asarray(vectors, dtype=np.float32), feature_names

def first_key(data, candidates, required=True):
    for key in candidates:
        if key in data:
            return np.asarray(data[key])
    if required:
        raise KeyError(f'None of keys found: {candidates}')
    return None

if RUN_MODE == 'SYNTHETIC':
    window_df, X_window, feature_names = make_synthetic_window_data()
    day36_gate = {
        'status': 'SYNTHETIC_CONTRACT_MODE',
        'hard_fatigue_diagnosis_allowed': False,
        'confidence_may_increase': False,
        'language_guard': 'pass',
    }
    input_sha256 = 'synthetic'
else:
    npz_path = PROFILE['npz']
    day36_path = PROFILE['day36_manifest']
    if not npz_path.exists():
        raise FileNotFoundError(npz_path)
    if not day36_path.exists():
        raise FileNotFoundError(f'Day36 manifest required in real mode: {day36_path}')
    day36_gate = json.loads(day36_path.read_text(encoding='utf-8'))
    if bool(day36_gate.get('hard_fatigue_diagnosis_allowed', False)):
        raise RuntimeError('Day36 contract illegally allows hard fatigue diagnosis.')
    if bool(day36_gate.get('confidence_may_increase', False)):
        raise RuntimeError('Day36 contract illegally allows confidence increase.')
    with np.load(npz_path, allow_pickle=False) as data:
        X_window = np.asarray(data['X'], dtype=np.float32)
        feature_names = first_key(data, ['feature_names', 'column_names']).astype(str).tolist()
        n = X_window.shape[0]
        subject_id = first_key(data, ['subject_id', 'groups']).astype(str)
        repetition_id = first_key(data, ['repetition_id', 'trial_id', 'record_id']).astype(str)
        task_label = first_key(data, ['y', 'label', 'canonical_label']).astype(str)
        split_name = first_key(data, ['split_names', 'partition', 'split']).astype(str)
        session_id = first_key(data, ['session_id', 'day_id'], required=False)
        if session_id is None:
            session_id = np.repeat('session-1', n)
        window_id = first_key(data, ['window_id'], required=False)
        if window_id is None:
            window_id = np.asarray([f'row-{i}' for i in range(n)])
        window_df = pd.DataFrame({
            'dataset_id': PROFILE['dataset_id'],
            'subject_id': subject_id,
            'session_id': session_id.astype(str),
            'protocol_id': 'dataset-primary4',
            'protocol_version': '1.0.0',
            'task_label': task_label,
            'repetition_id': repetition_id,
            'window_id': window_id.astype(str),
            'split_name': split_name,
            'quality_status': 'pass',
        })
    input_sha256 = sha256_file(npz_path)

if len(window_df) != len(X_window):
    raise RuntimeError('Metadata row count and X row count mismatch.')
if len(feature_names) != X_window.shape[1]:
    raise RuntimeError('Feature-name count and X dimension mismatch.')
if window_df['dataset_id'].nunique() != 1:
    raise RuntimeError('Pooled datasets detected.')
if window_df['repetition_id'].isna().any():
    raise RuntimeError('Missing repetition IDs.')

print({
    'rows': len(window_df),
    'features': X_window.shape[1],
    'subjects': window_df['subject_id'].nunique(),
    'repetitions': window_df['repetition_id'].nunique(),
    'sessions': window_df['session_id'].nunique(),
})

{'rows': 5760, 'features': 42, 'subjects': 12, 'repetitions': 480, 'sessions': 2}


## Cell 5 — Canonical repetition table and metric registry

**Input:** window-level features.  
**Output:** one row per repetition and frozen metric registry.

In [5]:
import yaml

meta_columns = [
    'dataset_id', 'subject_id', 'session_id', 'protocol_id',
    'protocol_version', 'task_label', 'repetition_id',
    'split_name', 'quality_status',
]
feature_frame = pd.DataFrame(X_window, columns=feature_names)
combined = pd.concat([window_df.reset_index(drop=True), feature_frame], axis=1)
rep_meta = combined.groupby('repetition_id', as_index=False)[meta_columns].first()
rep_features = combined.groupby('repetition_id', as_index=False)[feature_names].median()
rep_counts = combined.groupby('repetition_id', as_index=False).size().rename(columns={'size': 'valid_window_count'})
repetition_df = rep_meta.merge(rep_features, on='repetition_id').merge(rep_counts, on='repetition_id')

if repetition_df['repetition_id'].duplicated().any():
    raise RuntimeError('Duplicate repetition rows after aggregation.')
if not np.isfinite(repetition_df[feature_names].to_numpy()).all():
    raise RuntimeError('Non-finite repetition feature matrix.')

try:
    repetition_df.to_parquet(PATHS['repetition_table'], index=False)
    repetition_table_format = 'parquet'
except ImportError:
    fallback = PATHS['repetition_table'].with_suffix('.csv.gz')
    repetition_df.to_csv(fallback, index=False, compression='gzip')
    PATHS['repetition_table'] = fallback
    repetition_table_format = 'csv.gz_fallback_no_parquet_engine'
write_json(PATHS['repetition_manifest'], {
    'schema_version': 'taskc-repetition-feature-manifest.v2',
    'created_at_utc': utc_now_iso(),
    'dataset_id': PROFILE['dataset_id'],
    'input_sha256': input_sha256,
    'aggregation': 'median_across_valid_windows',
    'row_unit': 'repetition',
    'storage_format': repetition_table_format,
    'feature_count': len(feature_names),
    'feature_columns': feature_names,
    'repetition_count': len(repetition_df),
    'subject_count': int(repetition_df['subject_id'].nunique()),
    'session_count': int(repetition_df['session_id'].nunique()),
    'pooled_datasets': False,
})

metric_registry = {
    'schema_version': 'taskc-metric-registry.v2',
    'epsilon': EPS,
    'aggregation_unit': 'subject_session_protocol_class_repetition',
    'repeatability': {
        'absolute_difference': {'minimum_repetitions': 2},
        'symmetric_relative_difference': {'minimum_repetitions': 2},
        'within_subject_cv_percent': {'minimum_repetitions': 3, 'requires_positive_ratio_scale': True},
        'robust_mad_ratio': {'minimum_repetitions': 3},
        'icc_a1': {'implementation': 'standalone_two_way_random_absolute_agreement_single_measure_with_subject_bootstrap_ci', 'not_from_windows': True},
        'bland_altman': {'requires_paired_measurements': True},
    },
    'similarity': {
        'cosine': {'block_state': 'ZERO_NORM_VECTOR'},
        'pearson': {'block_state': 'CONSTANT_VECTOR'},
        'normalized_euclidean': {'requires_frozen_scaler': True},
        'universal_abnormal_threshold_allowed': False,
    },
    'cocontraction': {
        'requires_verified_anatomical_mapping': True,
        'requires_synchronized_envelopes': True,
        'requires_declared_normalization': True,
        'ineligible_value_is_zero': False,
    },
    'prohibited_inference': [
        'hard_fatigue_diagnosis', 'clinical_normality_claim',
        'pathology_claim', 'treatment_recommendation',
    ],
}
PATHS['registry'].write_text(yaml.safe_dump(metric_registry, sort_keys=False, allow_unicode=True), encoding='utf-8')
print('[PASS] Repetition table and metric registry written.')

[PASS] Repetition table and metric registry written.


## Cell 6 — Repeatability and ICC(A,1)

**Input:** repetition rows grouped by subject/session/class.  
**Output:** AD, SRD, CV/rMAD and cohort ICC where eligible.

In [6]:
ratio_scale_suffixes = {'MAV', 'RMS', 'WL', 'WAMP', 'VAR', 'IEMG', 'MNF', 'MDF', 'PKF'}
selected_scalar_features = [
    name for name in feature_names
    if name.split('__')[-1].upper() in {'MAV', 'RMS', 'MNF', 'MDF'}
]
if not selected_scalar_features:
    selected_scalar_features = feature_names[:min(8, len(feature_names))]

group_cols = ['subject_id', 'session_id', 'protocol_id', 'protocol_version', 'task_label']
repeat_rows = []
for keys, group in repetition_df.groupby(group_cols, sort=True):
    group = group.sort_values('repetition_id')
    for feature in selected_scalar_features:
        values = group[feature].to_numpy(float)
        cv, cv_state = within_subject_cv(values, feature.split('__')[-1].upper() in ratio_scale_suffixes)
        rmad, rmad_state = robust_mad_ratio(values)
        pairs = list(itertools.combinations(range(len(group)), 2))
        if not pairs:
            pairs = [(None, None)]
        for i, j in pairs:
            if i is None:
                pair_payload = {
                    'reference_repetition_id': None,
                    'query_repetition_id': None,
                    'absolute_difference': None,
                    'symmetric_relative_difference': None,
                    'pair_supportability': 'INSUFFICIENT_REPETITIONS',
                }
            else:
                srd, srd_state = symmetric_relative_difference(values[i], values[j])
                pair_payload = {
                    'reference_repetition_id': group.iloc[i]['repetition_id'],
                    'query_repetition_id': group.iloc[j]['repetition_id'],
                    'absolute_difference': absolute_difference(values[i], values[j]),
                    'symmetric_relative_difference': srd,
                    'pair_supportability': srd_state,
                }
            repeat_rows.append({
                **dict(zip(group_cols, keys)),
                'metric_id': feature,
                'repetition_count': len(group),
                'within_subject_cv_percent': cv,
                'cv_supportability': cv_state,
                'robust_mad_ratio': rmad,
                'rmad_supportability': rmad_state,
                **pair_payload,
            })

repeatability_df = pd.DataFrame(repeat_rows)
repeatability_df.to_csv(PATHS['repeatability'], index=False)
repeatability_summary = (
    repeatability_df.groupby(['subject_id', 'session_id', 'task_label'], as_index=False)
    .agg(
        median_srd=('symmetric_relative_difference', 'median'),
        median_cv_percent=('within_subject_cv_percent', 'median'),
        median_rmad=('robust_mad_ratio', 'median'),
        supported_pairs=('pair_supportability', lambda s: int((s == 'SUPPORTED').sum())),
    )
)
repeatability_summary.to_csv(PATHS['repeatability_summary'], index=False)

icc_rows = []
for session_id in repetition_df['session_id'].unique():
    for label in repetition_df['task_label'].unique():
        sub = repetition_df[(repetition_df['session_id'] == session_id) & (repetition_df['task_label'] == label)].copy()
        counts = sub.groupby('subject_id')['repetition_id'].count()
        if len(counts) < 3 or counts.min() < 2:
            continue
        r = int(counts.min())
        sub['rep_order'] = sub.groupby('subject_id').cumcount()
        sub = sub[sub['rep_order'] < r]
        for feature in selected_scalar_features:
            long = sub[['subject_id', 'rep_order', feature]].rename(
                columns={'subject_id': 'targets', 'rep_order': 'raters', feature: 'ratings'}
            )
            matrix = (
                sub.pivot(index='subject_id', columns='rep_order', values=feature)
                .sort_index(axis=0)
                .sort_index(axis=1)
                .to_numpy(float)
            )
            point, ci_low, ci_high, icc_state = icc_subject_bootstrap_ci(matrix)
            icc_rows.append({
                'session_id': session_id,
                'task_label': label,
                'metric_id': feature,
                'icc_type': 'ICC(A,1)',
                'icc_value': point,
                'icc_ci_low': ci_low,
                'icc_ci_high': ci_high,
                'icc_supportability': icc_state,
                'subjects': int(len(counts)),
                'repetitions_per_subject': r,
                'ci_method': 'subject_cluster_bootstrap',
            })
pd.DataFrame(icc_rows).to_csv(PATHS['icc'], index=False)
print(repeatability_summary.head())

  subject_id session_id       task_label  median_srd  median_cv_percent  \
0        S01         D1       hand_close    0.352185          13.258437   
1        S01         D1             rest    0.369344          21.910164   
2        S01         D1  wrist_extension    0.441838          15.897434   
3        S01         D1    wrist_flexion    0.506730          36.634989   
4        S01         D2       hand_close    0.278180          13.710084   

   median_rmad  supported_pairs  
0     0.362862              120  
1     0.484139              120  
2     0.380190              120  
3     0.458519              120  
4     0.171956              120  


## Cell 7 — Similarity with train-only frozen scaler

**Input:** repetition feature vectors.  
**Output:** within-subject same-class pair similarities.

In [7]:
train_mask = repetition_df['split_name'].astype(str).str.lower().eq('train')
if not train_mask.any():
    raise RuntimeError('No training repetitions available for frozen scaler.')

scaler = StandardScaler().fit(repetition_df.loc[train_mask, feature_names])
Z = scaler.transform(repetition_df[feature_names])
if not np.isfinite(Z).all():
    raise RuntimeError('Non-finite scaled repetition vectors.')

similarity_rows = []
for keys, group in repetition_df.groupby(group_cols, sort=True):
    idx = group.index.to_numpy()
    for i_pos, j_pos in itertools.combinations(range(len(group)), 2):
        i, j = idx[i_pos], idx[j_pos]
        x, y = Z[i], Z[j]
        cosine, cosine_state = cosine_similarity_safe(x, y)
        pearson, pearson_state = pearson_similarity_safe(x, y)
        supportability = 'SUPPORTED' if cosine_state == pearson_state == 'SUPPORTED' else ';'.join(sorted({cosine_state, pearson_state}))
        similarity_rows.append({
            **dict(zip(group_cols, keys)),
            'reference_repetition_id': repetition_df.loc[i, 'repetition_id'],
            'query_repetition_id': repetition_df.loc[j, 'repetition_id'],
            'reference_scope': 'within_subject_same_session_same_class',
            'cosine_similarity': cosine,
            'pearson_correlation': pearson,
            'normalized_euclidean_distance': normalized_euclidean(x, y),
            'feature_arm': 'ALL14',
            'scaler_fit_partition': 'train_only',
            'supportability': supportability,
        })

similarity_df = pd.DataFrame(similarity_rows)
similarity_df.to_csv(PATHS['similarity'], index=False)
similarity_summary = (
    similarity_df.groupby(['subject_id', 'session_id', 'task_label'], as_index=False)
    .agg(
        median_cosine=('cosine_similarity', 'median'),
        median_pearson=('pearson_correlation', 'median'),
        median_normalized_euclidean=('normalized_euclidean_distance', 'median'),
        pair_count=('reference_repetition_id', 'count'),
    )
)
similarity_summary.to_csv(PATHS['similarity_summary'], index=False)
print(similarity_summary.head())

  subject_id session_id       task_label  median_cosine  median_pearson  \
0        S01         D1       hand_close       0.962411        0.963603   
1        S01         D1             rest       0.957652        0.958977   
2        S01         D1  wrist_extension       0.951788        0.953657   
3        S01         D1    wrist_flexion       0.958094        0.957996   
4        S01         D2       hand_close       0.963777        0.964153   

   median_normalized_euclidean  pair_count  
0                     0.306986          10  
1                     0.298361          10  
2                     0.298767          10  
3                     0.292483          10  
4                     0.300855          10  


## Cell 8 — Between-session agreement and co-contraction eligibility

**Input:** session summaries and optional verified mapping flags.  
**Output:** Bland–Altman evidence and explicit co-contraction state.

In [8]:
between_session_rows = []
sessions = sorted(repetition_df['session_id'].astype(str).unique())
if len(sessions) >= 2:
    s1, s2 = sessions[:2]
    for label in repetition_df['task_label'].unique():
        for feature in selected_scalar_features:
            a = repetition_df[(repetition_df['session_id'].astype(str) == s1) & (repetition_df['task_label'] == label)].groupby('subject_id')[feature].median()
            b = repetition_df[(repetition_df['session_id'].astype(str) == s2) & (repetition_df['task_label'] == label)].groupby('subject_id')[feature].median()
            common = sorted(set(a.index) & set(b.index))
            between_session_rows.append({
                'session_a': s1,
                'session_b': s2,
                'task_label': label,
                'metric_id': feature,
                **bland_altman(a.loc[common], b.loc[common]),
            })
pd.DataFrame(between_session_rows).to_csv(PATHS['bland_altman'], index=False)

MUSCLE_MAPPING_VERIFIED = bool(int(os.getenv('DAY37_MUSCLE_MAPPING_VERIFIED', '0')))
SYNC_ENVELOPES_AVAILABLE = bool(int(os.getenv('DAY37_SYNC_ENVELOPES_AVAILABLE', '0')))
NORMALIZATION_DECLARED = bool(int(os.getenv('DAY37_NORMALIZATION_DECLARED', '0')))

eligibility_checks = {
    'verified_agonist_antagonist_mapping': MUSCLE_MAPPING_VERIFIED,
    'synchronized_time_base': SYNC_ENVELOPES_AVAILABLE,
    'normalization_method_declared': NORMALIZATION_DECLARED,
}
if all(eligibility_checks.values()):
    cocontraction_status = 'SUPPORTED_WITH_EXPLICIT_MAPPING'
    t = np.linspace(0, 5, 1000, endpoint=False)
    a = np.clip(0.2 + 0.8 * np.sin(np.pi * t / 5) ** 2, 0, None)
    b = np.clip(0.15 + 0.55 * np.sin(np.pi * t / 5 + 0.4) ** 2, 0, None)
    metrics = cocontraction_metrics(a, b, dt=t[1] - t[0], threshold_a=0.3, threshold_b=0.3)
    cocontraction_df = pd.DataFrame([{
        'cocontraction_id': 'synthetic-verified-pair-1',
        'subject_id': 'SYN',
        'session_id': 'SYN',
        'protocol_id': 'synthetic-cocontraction',
        'repetition_id': 'SYN-R01',
        'pair_id': 'PAIR-VERIFIED-01',
        'agonist_muscle_id': 'MUSCLE_A',
        'antagonist_muscle_id': 'MUSCLE_B',
        'side': 'unspecified',
        'normalization_method': 'synthetic_reference',
        'supportability': 'SUPPORTED',
        **metrics,
    }])
else:
    cocontraction_status = 'NOT_ELIGIBLE_ANATOMICAL_MAPPING_UNVERIFIED'
    cocontraction_df = pd.DataFrame(columns=[
        'cocontraction_id', 'subject_id', 'session_id', 'protocol_id',
        'repetition_id', 'pair_id', 'supportability',
    ])

cocontraction_df.to_csv(PATHS['cocontraction_results'], index=False)
write_json(PATHS['cocontraction_eligibility'], {
    'schema_version': 'cocontraction-eligibility.v2',
    'dataset_id': PROFILE['dataset_id'],
    'status': cocontraction_status,
    'checks': eligibility_checks,
    'numeric_zero_used_for_ineligible': False,
    'reason_codes': [] if all(eligibility_checks.values()) else [k.upper() for k, v in eligibility_checks.items() if not v],
    'prohibited_inference': ['pathological_cocontraction', 'fatigue_diagnosis'],
})
print(json.loads(PATHS['cocontraction_eligibility'].read_text()))

{'schema_version': 'cocontraction-eligibility.v2', 'dataset_id': 'synthetic-taskc-contract', 'status': 'NOT_ELIGIBLE_ANATOMICAL_MAPPING_UNVERIFIED', 'checks': {'verified_agonist_antagonist_mapping': False, 'synchronized_time_base': False, 'normalization_method_declared': False}, 'numeric_zero_used_for_ineligible': False, 'reason_codes': ['VERIFIED_AGONIST_ANTAGONIST_MAPPING', 'SYNCHRONIZED_TIME_BASE', 'NORMALIZATION_METHOD_DECLARED'], 'prohibited_inference': ['pathological_cocontraction', 'fatigue_diagnosis']}


## Cell 9 — Sensitivity, Day 36 integration, final gate and handoff

**Input:** all generated artifacts.  
**Output:** final manifest and `day37-taskc-handoff.zip`.

In [9]:
write_json(PATHS['sensitivity'], {
    'schema_version': 'taskc-sensitivity-report.v2',
    'created_at_utc': utc_now_iso(),
    'epsilon_values_reviewed': [1e-12, 1e-9, 1e-6],
    'window_independence_claim': False,
    'repetition_aggregation': 'median',
    'scaler_fit_partition': 'train_only',
    'universal_similarity_threshold': None,
    'co_contraction_status': cocontraction_status,
    'limitations': [
        'Task C values are descriptive engineering evidence.',
        'Repeatability is not clinical normality.',
        'Similarity is not pathology.',
        'Co-contraction is blocked without verified anatomy and synchronization.',
    ],
})

write_json(PATHS['integration'], {
    'schema_version': 'day37-day36-integration.v2',
    'hard_fatigue_diagnosis_created': False,
    'confidence_increase_allowed': False,
    'quality_fail_semantics': 'QUALITY_BLOCKED_NOT_NO_FATIGUE',
    'events': [
        {
            'metric_family': 'repeatability',
            'metric_id': 'median_srd',
            'supportability': 'SUPPORTED',
            'context_lane': 'performance_consistency',
            'maximum_context_effect': 'POSSIBLE',
        },
        {
            'metric_family': 'cocontraction',
            'metric_id': 'mean_cci',
            'supportability': cocontraction_status,
            'context_lane': 'ignored_with_reason' if cocontraction_status != 'SUPPORTED_WITH_EXPLICIT_MAPPING' else 'performance_consistency',
            'maximum_context_effect': 'NONE' if cocontraction_status != 'SUPPORTED_WITH_EXPLICIT_MAPPING' else 'POSSIBLE',
        },
    ],
})

required = [p for name, p in PATHS.items() if name not in {'final', 'handoff'}]
checks = {
    'single_dataset_only': bool(repetition_df['dataset_id'].nunique() == 1),
    'repetition_unit_preserved': bool(repetition_df['repetition_id'].is_unique),
    'no_nonfinite_features': bool(np.isfinite(repetition_df[feature_names]).all().all()),
    'train_only_scaler': True,
    'co_contraction_fail_closed': bool(
        cocontraction_status == 'SUPPORTED_WITH_EXPLICIT_MAPPING' or cocontraction_status.startswith('NOT_ELIGIBLE')
    ),
    'hard_fatigue_diagnosis_absent': True,
    'all_required_artifacts_exist': bool(all(p.exists() for p in required)),
}
failed = [k for k, v in checks.items() if not v]
status = 'BLOCKED_WITH_EVIDENCE' if failed else (
    'GO_FOR_DAY38_WITH_METRIC_LIMITATIONS' if cocontraction_status.startswith('NOT_ELIGIBLE')
    else 'GO_FOR_DAY38_TASK_C_VALIDATION'
)
final_manifest = {
    'schema_version': 'day37-final-manifest.v2',
    'created_at_utc': utc_now_iso(),
    'dataset_id': PROFILE['dataset_id'],
    'run_mode': RUN_MODE,
    'status': status,
    'checks': checks,
    'failed_checks': failed,
    'sealed_test_opened': SEALED_TEST_OPENED,
    'hard_fatigue_diagnosis_allowed': False,
    'confidence_may_increase': False,
    'artifacts': {p.name: {'path': str(p), 'sha256': sha256_file(p)} for p in required},
}
write_json(PATHS['final'], final_manifest)

if failed:
    raise RuntimeError(json.dumps(final_manifest, indent=2))

if PATHS['handoff'].exists():
    PATHS['handoff'].unlink()
tmp_archive_base = OUTPUT_DIR.parent / f'.{PATHS["handoff"].stem}-build'
tmp_zip = Path(shutil.make_archive(str(tmp_archive_base), 'zip', root_dir=OUTPUT_DIR))
shutil.move(str(tmp_zip), str(PATHS['handoff']))
assert PATHS['handoff'].exists()

print('=' * 88)
print('[DAY37 FINAL]')
print(json.dumps(final_manifest, indent=2))
print('Handoff:', PATHS['handoff'])

[DAY37 FINAL]
{
  "schema_version": "day37-final-manifest.v2",
  "created_at_utc": "2026-08-04T08:08:09.940865+00:00",
  "dataset_id": "synthetic-taskc-contract",
  "run_mode": "SYNTHETIC",
  "status": "GO_FOR_DAY38_WITH_METRIC_LIMITATIONS",
  "checks": {
    "single_dataset_only": true,
    "repetition_unit_preserved": true,
    "no_nonfinite_features": true,
    "train_only_scaler": true,
    "co_contraction_fail_closed": true,
    "hard_fatigue_diagnosis_absent": true,
    "all_required_artifacts_exist": true
  },
  "failed_checks": [],
  "sealed_test_opened": false,
  "hard_fatigue_diagnosis_allowed": false,
  "confidence_may_increase": false,
  "artifacts": {
    "taskc-metric-registry.yaml": {
      "path": "/content/day37-taskc-synthetic-v2/taskc-metric-registry.yaml",
      "sha256": "828aa750706ffd7ff1245a13338473636b5be18c1ba512dfd7eb7c08878be183"
    },
    "taskc-repetition-features.parquet": {
      "path": "/content/day37-taskc-synthetic-v2/taskc-repetition-features.parqu

In [10]:
from google.colab import files

# Tải tệp handoff ZIP về máy
files.download('/content/day37-taskc-synthetic-v2/day37-taskc-handoff.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>